# HW5: Fixed-Income Valuation and First-Order Risk

All five library tasks are completed. This notebook contains executed outputs.

Written answers: [Part I — Implicit Function Differentiation](docs/part_i_implicit_differentiation.md).
Implementation details: [Method and validation](docs/implementation.md).

In [1]:
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfCreateProductFixedAccruedCashflow,
    qfCreateValuationParameters,
    qfCreateValuationParametersCollection,
    qfCreateValueReport,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import IndexRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve import YieldCurveModelComponent
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductFixedAccrued

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells.
The funding component adds an IFR spread over SOFR.

In [2]:
VALUE_DATE = "2026-01-02"
PAYMENT_DATE = Date("2026-10-02")
FUNDING_INDEX = "SOFR-1B-FLAT"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-02", "2028-01-02"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2027-01-02"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_quotes.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Node date,
2026-04-02,0.030000
2027-01-02,0.040000
2028-01-02,0.050000


,SOFR-1B-FLAT IFR spread
Node date,
2026-07-02,0.000000
2027-01-02,0.001000


## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first and funding second.

In [3]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, funding_method])

## 3. Build the model

Copy IFR inputs into the model. The registered funding component references SOFR.

In [5]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
print("Component sizes:", model.num_sub_components)

Component sizes: [3, 2]


The following helpers retrieve a component (`retrieve_model_component`), allocate zero-filled
gradient blocks in component order (`resize_gradient`), and locate a component's block
(`retrieve_model_component_index`, using the `component_indices` map).

In [6]:
index = IndexRegistry().get("SOFR-1B")
component = cast(YieldCurveModelComponent, model.retrieve_model_component(index))
funding_component = model.retrieve_model_component(FUNDING_INDEX)

print("SOFR-1B ->", type(component).__name__, "|", component.component_identifier.name())
print("SOFR-1B-FLAT ->", funding_component.component_identifier.name())
display(pd.DataFrame(component.state_data.T, columns=["Node time (years)", "IFR"]))

SOFR-1B -> YieldCurveModelComponent | SOFRON Actual/360
SOFR-1B-FLAT -> SOFR-1B-FLAT


,Node time (years),IFR
0,0.246575,0.030000
1,1.000000,0.040000
2,1.999993,0.050000


In [7]:
gradient = []
model.resize_gradient(gradient)
print("[SOFR block, funding block]:", gradient)

[SOFR block, funding block]: [array([0., 0., 0.]), array([0., 0.])]


In [8]:
print("Stored map:", model.component_indices)
print("SOFR-1B -> block", model.retrieve_model_component_index("SOFR-1B"))
print("SOFR-1B-FLAT -> block", model.retrieve_model_component_index(FUNDING_INDEX))

Stored map: {'SOFRON Actual/360': 0, 'SOFR-1B-FLAT': 1}
SOFR-1B -> block 0
SOFR-1B-FLAT -> block 1


## 4. Discount-factor gradients

Completed **Task 1** in `YieldCurveModelComponent` and **Task 2** in `YieldCurve`
(`fixedincomelib/yield_curve/yield_curve_model.py`). Calculate the local derivative, then apply
the product rule recursively to $D_{\mathrm{funding}}=D_{\mathrm{spread}}D_{\mathrm{SOFR}}$.

Gradient methods write raw derivatives into the supplied container and return `None`.
`scaler` scales the new contribution; `accumulate=False` replaces the output and `True` adds to it.

With the default piecewise-constant interpolation, the third SOFR segment starts after
2027-01-02. Its gradient is zero for the 2026-10-02 payment.

In [9]:
local_gradient = np.zeros(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_gradient)
print("Local SOFR discount-factor gradient:", local_gradient)

Local SOFR discount-factor gradient: [-0.239898 -0.487793 -0.      ]


In [10]:
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, gradient)
print("Model gradient [SOFR block, funding block]:", gradient)

Model gradient [SOFR block, funding block]: [array([-0.239838, -0.48767 ,  0.      ]), array([-0.482341, -0.245168])]


## 5. Fixed-accrued valuation engine

Completed **Tasks 3–5** in `ValuationEngineProductFixedAccrued`: `__init__`, `calculate_value`,
and `calculate_risk` (`fixedincomelib/yield_curve/valuation_engine.py`).
Select the funding curve, then compute $C=N\alpha$, $PV=C D(T_p)$ and
$\nabla PV=C\nabla D(T_p)$. Notional is signed; this product has no separate fixed-rate field.

| Payment date | PV / cash today | Risk |
|---|---|---|
| Future | $C D(T_p)$ / 0 | $C\nabla D(T_p)$ |
| Today | $C$ / $C$ | 0 |
| Past | 0 / 0 | 0 |

In [11]:
product = qfCreateProductFixedAccruedCashflow(
    "2026-04-02", "2026-07-02", "USD", 1_000_000, "ACT/360",
    payment_date=PAYMENT_DATE.ISO(), business_day_convention="NONE",
)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Accrual fraction:", product.accrued)
print("Cash amount:", product.notional * product.accrued)

Accrual fraction: 0.25277777777777777
Cash amount: 252777.77777777778


In [12]:
engine = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
engine.calculate_value()

product_gradient = []
engine.calculate_risk(product_gradient)

print("PV:", engine.value)
print("Cash today:", engine.cash)
print("Product gradient [SOFR block, funding block]:", product_gradient)

PV: 245870.91057967892
Cash today: 0.0
Product gradient [SOFR block, funding block]: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


## 6. Reports

`qfCreateValueReport` calls the engine for `PV` or `FirstOrderRisk`.
`state_gradient` retains raw blocks; `display()` multiplies risk by `0.0001`
to show the signed PV change per positive 1 bp node shift.

In [13]:
print("PV report:", qfCreateValueReport(model, product, parameters, "PV"))
risk_report = qfCreateValueReport(model, product, parameters, "FirstOrderRisk")

print("Raw blocks:", risk_report.state_gradient)
display(risk_report.display()[["COMPONENT", "NODE_INDEX", "AXIS1", "UNIT", "VALUES"]])

PV report: [['USD', 245870.91057967892]]
Raw blocks: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


,COMPONENT,NODE_INDEX,AXIS1,UNIT,VALUES
0,SOFRON Actual/360,0,2026-04-02,0.000100,-6.062570
1,SOFRON Actual/360,1,2027-01-02,0.000100,-12.327226
2,SOFRON Actual/360,2,2028-01-02,0.000100,0.000000
3,SOFR-1B-FLAT,0,2026-07-02,0.000100,-12.192503
4,SOFR-1B-FLAT,1,2027-01-02,0.000100,-6.197294
